# AI Programming — Lecture 2
## AI를 위한 수학 (Mathematics for AI)

이 노트북에서는 강의에서 다룬 **선형대수(Linear Algebra)**와
**확률·통계(Probability & Statistics)**의 핵심 개념을 NumPy로 직접 확인합니다.

### 학습 목표
실습을 마치면 다음 내용을 코드로 확인하고 설명할 수 있어야 합니다.

- scalar, vector, matrix, tensor의 차이와 `shape`를 이해합니다.
- transpose, matrix multiplication, element-wise product의 차이를 구분합니다.
- broadcasting과 vector concatenation을 사용할 수 있습니다.
- dot product, outer product, norm, cosine similarity를 계산할 수 있습니다.
- 선형변환과 affine transformation을 행렬 연산으로 표현할 수 있습니다.
- eigenvalue와 eigenvector의 의미를 간단한 예제로 확인할 수 있습니다.
- determinant의 기하학적 의미를 확인할 수 있습니다.
- softmax를 직접 구현하고 확률 벡터의 성질을 확인할 수 있습니다.
- conditional, joint, marginal probability와 Bayes' theorem을 계산할 수 있습니다.
- expectation, variance, covariance를 계산할 수 있습니다.

### 실습 방법
1. 셀을 **위에서부터 순서대로 실행**하세요.
2. 각 절의 **확인할 내용**을 읽고 출력이나 그래프를 해석하세요.
3. `TODO`가 있는 부분은 값을 직접 바꾸어 다시 실행하세요.
4. 단순히 결과를 확인하는 것보다 **shape가 어떻게 변하는지**를 항상 확인하세요.

> 그래프의 축과 제목은 실행 환경의 한글 폰트 문제를 피하기 위해 일부 영어로 표시합니다.

## 0. 라이브러리 불러오기

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

# Part I. Linear Algebra

## 1. Scalar, Vector, Matrix, Tensor

딥러닝에서는 대부분의 데이터와 파라미터가 배열 형태로 표현됩니다.

- **Scalar**: 하나의 값
- **Vector**: 1차원 배열
- **Matrix**: 2차원 배열
- **Tensor**: 일반적인 다차원 배열

NumPy에서는 `ndim`과 `shape`를 이용해 차원을 확인할 수 있습니다.

In [ ]:
scalar = np.array(3.0)
vector = np.array([1.0, 2.0, 3.0])
matrix = np.array([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0]
])
tensor = np.zeros((2, 3, 4))

for name, x in [
    ("scalar", scalar),
    ("vector", vector),
    ("matrix", matrix),
    ("tensor", tensor),
]:
    print(f"{name:>6} | ndim = {x.ndim} | shape = {x.shape}")

### 확인할 내용

- scalar의 `shape`는 `()`입니다.
- 길이 3인 vector의 `shape`는 `(3,)`입니다.
- 2행 3열 matrix의 `shape`는 `(2, 3)`입니다.
- `(2, 3, 4)` tensor는 세 개의 축(axis)을 가집니다.

> **중요:** 딥러닝 코드에서 오류의 상당수는 값 자체보다 `shape`가 맞지 않아서 발생합니다.

## 2. Transpose와 기본 벡터

Transpose는 행과 열을 교환합니다.

$$
\mathbf{A}
\in \mathbb{R}^{m \times n}
\quad \Rightarrow \quad
\mathbf{A}^{\top}
\in \mathbb{R}^{n \times m}
$$

NumPy에서는 `.T`를 사용합니다.

In [ ]:
A = np.array([
    [1, 2, 3],
    [4, 5, 6]
])

print("A:")
print(A)
print("shape:", A.shape)

print("\nA.T:")
print(A.T)
print("shape:", A.T.shape)

print("\nZero vector:", np.zeros(4))
print("One vector :", np.ones(4))
print("Unit vector e_2:", np.eye(4)[1])

### 확인할 내용

- `(2, 3)` matrix를 transpose하면 `(3, 2)`가 됩니다.
- `np.eye(n)`의 각 행 또는 열은 standard basis에 해당합니다.

## 3. Linear Combination, Dot Product, Outer Product

두 벡터 $\mathbf{v}_1$, $\mathbf{v}_2$의 선형결합은

$$
\mathbf{x}
=
a_1\mathbf{v}_1 + a_2\mathbf{v}_2
$$

와 같이 표현할 수 있습니다.

Dot product는 두 벡터를 하나의 scalar로 축약합니다.

$$
\mathbf{x}^{\top}\mathbf{y}
=
\sum_i x_i y_i
$$

Outer product는 두 벡터로 matrix를 만듭니다.

$$
\mathbf{x}\mathbf{y}^{\top}
$$

In [ ]:
v1 = np.array([1.0, 0.0])
v2 = np.array([0.0, 1.0])

a1, a2 = 2.0, 3.0
x = a1 * v1 + a2 * v2

u = np.array([1.0, 3.0, 2.0])
v = np.array([1.0, 0.0, 2.0])

dot_uv = u @ v
outer_uv = np.outer(u, v)

print("Linear combination:", x)
print("\nDot product:", dot_uv)
print("\nOuter product:")
print(outer_uv)
print("Outer product shape:", outer_uv.shape)

### 직접 해보기 1

`u`, `v`를 다른 값으로 바꾸어 다음을 확인하세요.

1. `u @ v`의 결과 shape는 무엇입니까?
2. `np.outer(u, v)`의 결과 shape는 무엇입니까?
3. 두 연산이 같은 원소들을 사용하지만 전혀 다른 결과를 만드는 이유는 무엇입니까?

## 4. Matrix Multiplication vs. Element-wise Product

Matrix multiplication:

$$
(m \times p)(p \times n)
\rightarrow
(m \times n)
$$

NumPy에서는 `A @ B` 또는 `np.matmul(A, B)`를 사용합니다.

Element-wise(Hadamard) product는 **같은 위치의 원소끼리** 곱합니다.

$$
\mathbf{A} \odot \mathbf{B}
$$

NumPy에서는 `A * B`를 사용합니다.

In [ ]:
A = np.array([
    [1, 2],
    [3, 4]
])

B = np.array([
    [5, 6],
    [7, 8]
])

print("A @ B:")
print(A @ B)

print("\nA * B:")
print(A * B)

> ### ✅ 체크포인트
> `@`와 `*`는 전혀 다른 연산입니다.  
> 신경망의 layer에서 사용하는 가중치 연산은 주로 **matrix multiplication**입니다.

## 5. Broadcasting과 Concatenation

Broadcasting은 서로 다른 shape의 배열을 element-wise 연산할 때
NumPy가 호환 가능한 축을 자동으로 확장하는 기능입니다.

Concatenation은 여러 벡터 또는 배열을 이어 붙이는 연산입니다.

In [ ]:
X = np.array([
    [1, 2, 3],
    [4, 5, 6]
])

bias = np.array([10, 20, 30])

print("X shape:", X.shape)
print("bias shape:", bias.shape)

print("\nX + bias:")
print(X + bias)

a = np.array([1, 2, 3])
b = np.array([4, 5])

c = np.concatenate([a, b])

print("\nConcatenated vector:", c)
print("shape:", c.shape)

### 확인할 내용

`bias`의 shape는 `(3,)`이지만 `(2, 3)` matrix의 각 행에 자동으로 더해집니다.

이 구조는 신경망의 affine transformation

$$
\mathbf{y} = \mathbf{W}\mathbf{x} + \mathbf{b}
$$

에서 bias를 batch 전체에 더할 때 반복적으로 사용됩니다.

## 6. Linear Transformation과 Affine Transformation

Linear transformation:

$$
\mathbf{y} = \mathbf{W}\mathbf{x}
$$

Affine transformation:

$$
\mathbf{y} = \mathbf{W}\mathbf{x} + \mathbf{b}
$$

신경망의 `Dense` layer는 기본적으로 affine transformation입니다.

In [ ]:
x = np.array([2.0, 1.0])

W = np.array([
    [2.0, 0.5],
    [-1.0, 1.0]
])

b = np.array([1.0, 2.0])

y_linear = W @ x
y_affine = W @ x + b

print("x =", x)
print("W @ x =", y_linear)
print("W @ x + b =", y_affine)

### 직접 해보기 2

`b = [0, 0]`으로 바꾸면 affine transformation과 linear transformation은 어떻게 됩니까?

또한 `x = [1, 0]`, `x = [0, 1]`을 각각 입력해 보세요.  
이는 $\mathbf{W}$의 각 열이 공간에서 어떤 방향으로 변환되는지 보여줍니다.

## 7. Eigenvalue와 Eigenvector

Square matrix $\mathbf{W}$에 대해

$$
\mathbf{W}\mathbf{x}
=
\lambda \mathbf{x}
$$

를 만족하는 0이 아닌 벡터 $\mathbf{x}$를 eigenvector,
scalar $\lambda$를 eigenvalue라고 합니다.

즉, 변환 후에도 **방향은 유지되고 크기만 변하는 특별한 방향**입니다.

In [ ]:
W = np.array([
    [2.0, 1.0],
    [1.0, 2.0]
])

eigenvalues, eigenvectors = np.linalg.eig(W)

print("Eigenvalues:")
print(eigenvalues)

print("\nEigenvectors (columns):")
print(eigenvectors)

# 첫 번째 eigenpair 검증
lam = eigenvalues[0]
x = eigenvectors[:, 0]

print("\nW @ x:")
print(W @ x)

print("\nlambda * x:")
print(lam * x)

print("\nAre they close?", np.allclose(W @ x, lam * x))

### 확인할 내용

`np.linalg.eig()`가 반환하는 eigenvector는 **각 열(column)**에 저장됩니다.

`W @ x`와 `lambda * x`가 거의 같은지 확인하세요.

## 8. Norms

벡터의 크기를 측정하는 대표적인 norm은 다음과 같습니다.

### 1-Norm

$$
\|\mathbf{x}\|_1
=
\sum_i |x_i|
$$

### 2-Norm

$$
\|\mathbf{x}\|_2
=
\sqrt{\sum_i x_i^2}
$$

### Infinity Norm

$$
\|\mathbf{x}\|_{\infty}
=
\max_i |x_i|
$$

Matrix에서는 Frobenius norm도 자주 사용합니다.

$$
\|\mathbf{A}\|_F
=
\sqrt{\sum_{i,j} A_{ij}^2}
$$

In [ ]:
x = np.array([3.0, 4.0, -2.0])

print("1-norm       :", np.linalg.norm(x, ord=1))
print("2-norm       :", np.linalg.norm(x, ord=2))
print("infinity-norm:", np.linalg.norm(x, ord=np.inf))

A = np.array([
    [1.0, 2.0],
    [3.0, 4.0]
])

print("Frobenius norm:", np.linalg.norm(A, ord="fro"))

## 9. Cosine Similarity

두 벡터의 cosine similarity는

$$
\cos\theta
=
\frac{
\mathbf{x}^{\top}\mathbf{y}
}{
\|\mathbf{x}\|_2
\|\mathbf{y}\|_2
}
$$

로 정의됩니다.

크기보다 **방향의 유사성**을 측정합니다.

In [ ]:
def cosine_similarity(x, y):
    return (x @ y) / (
        np.linalg.norm(x) * np.linalg.norm(y)
    )

x = np.array([1.0, 1.0])
y_same = np.array([2.0, 2.0])
y_orthogonal = np.array([1.0, -1.0])
y_opposite = np.array([-1.0, -1.0])

print("same direction :", cosine_similarity(x, y_same))
print("orthogonal     :", cosine_similarity(x, y_orthogonal))
print("opposite       :", cosine_similarity(x, y_opposite))

### 확인할 내용

- 같은 방향 → 약 `1`
- 직교 → 약 `0`
- 반대 방향 → 약 `-1`

Cosine similarity는 이후 embedding이나 representation의 유사도를 비교할 때 자주 사용됩니다.

## 10. Determinant

2차원에서 determinant의 절댓값은 선형변환이 **면적을 얼마나 확대 또는 축소하는지** 나타냅니다.

$$
\mathbf{W}
=
\begin{bmatrix}
a & b \\
c & d
\end{bmatrix}
$$

이면

$$
\det(\mathbf{W})
=
ad-bc
$$

입니다.

In [ ]:
W = np.array([
    [2.0, 0.0],
    [0.0, 3.0]
])

det_W = np.linalg.det(W)

print("W:")
print(W)
print("\ndet(W) =", det_W)
print("|det(W)| =", abs(det_W))

### 확인할 내용

이 변환은 x 방향을 2배, y 방향을 3배 늘립니다.  
따라서 면적은 `2 × 3 = 6`배가 됩니다.

`|det(W)| = 6`인지 확인하세요.

## 11. Softmax Function

Softmax는 logits $\mathbf{x}$를 확률 벡터 $\mathbf{y}$로 변환합니다.

$$
y_i
=
\frac{e^{x_i}}
{\sum_j e^{x_j}}
$$

각 원소는 0과 1 사이이고, 전체 합은 1입니다.

In [ ]:
def softmax(x):
    # numerical stability를 위해 최댓값을 먼저 뺍니다.
    z = x - np.max(x)
    exp_z = np.exp(z)
    return exp_z / np.sum(exp_z)

logits = np.array([1.0, 2.0, 3.0, 4.0])
probs = softmax(logits)

print("logits:", logits)
print("probabilities:", probs)
print("sum:", probs.sum())

### 직접 해보기 3

`logits`의 마지막 값을 `4 → 10`으로 바꾸어 보세요.

- 해당 class의 확률은 어떻게 변합니까?
- 다른 class들의 확률은 어떻게 변합니까?
- 확률의 전체 합은 여전히 1입니까?

# Part II. Probability & Statistics

## 12. Conditional Probability와 Joint Probability

Conditional probability:

$$
P(A \mid B)
=
\frac{P(A \cap B)}
{P(B)}
$$

Joint probability:

$$
P(A,B)
=
P(A \cap B)
$$

간단한 주사위 시행을 이용하여 직접 계산해 봅니다.

In [ ]:
# 공정한 6면체 주사위
outcomes = np.array([1, 2, 3, 4, 5, 6])

# A: 짝수
A = outcomes % 2 == 0

# B: 3보다 큼
B = outcomes > 3

P_A = np.mean(A)
P_B = np.mean(B)
P_A_and_B = np.mean(A & B)
P_A_given_B = P_A_and_B / P_B

print("P(A) =", P_A)
print("P(B) =", P_B)
print("P(A and B) =", P_A_and_B)
print("P(A | B) =", P_A_given_B)

## 13. Marginal Probability와 Law of Total Probability

Joint distribution $P(X,Y)$에서 $Y$를 모두 더하면
$X$의 marginal probability를 얻습니다.

$$
P(X=x)
=
\sum_y P(X=x,Y=y)
$$

In [ ]:
# 행: X의 두 상태
# 열: Y의 세 상태
joint = np.array([
    [0.10, 0.20, 0.10],
    [0.15, 0.25, 0.20]
])

print("Joint distribution:")
print(joint)

P_X = joint.sum(axis=1)
P_Y = joint.sum(axis=0)

print("\nP(X):", P_X)
print("P(Y):", P_Y)
print("Total probability:", joint.sum())

### 확인할 내용

- joint distribution 전체 합은 1이어야 합니다.
- `axis=1`로 합하면 각 행에 대응하는 $P(X)$를 얻습니다.
- `axis=0`로 합하면 각 열에 대응하는 $P(Y)$를 얻습니다.

## 14. Bayes' Theorem

Bayes' theorem은 다음과 같습니다.

$$
P(D \mid T^+)
=
\frac{
P(T^+ \mid D)P(D)
}{
P(T^+)
}
$$

여기서

- $P(D)$: prior
- $P(T^+ \mid D)$: likelihood
- $P(T^+)$: evidence
- $P(D \mid T^+)$: posterior

### 예제
다음과 같은 검사를 생각해 봅니다.

- 질병 유병률: 1%
- 민감도: 99%
- 거짓 양성률: 5%

양성 판정을 받은 사람이 실제로 질병을 가지고 있을 확률을 계산합니다.

In [ ]:
P_D = 0.01
P_not_D = 1.0 - P_D

P_pos_given_D = 0.99
P_pos_given_not_D = 0.05

# Law of total probability
P_pos = (
    P_pos_given_D * P_D
    + P_pos_given_not_D * P_not_D
)

# Bayes' theorem
P_D_given_pos = (
    P_pos_given_D * P_D
    / P_pos
)

print("P(positive) =", P_pos)
print("P(disease | positive) =", P_D_given_pos)

### 확인할 내용

검사의 정확도가 높더라도 **prior(유병률)**가 매우 낮으면
양성 판정 후 실제 질병일 확률은 직관보다 낮을 수 있습니다.

> 확률을 해석할 때 likelihood만 보지 말고 prior와 함께 생각해야 합니다.

## 15. Expectation

Discrete random variable의 expectation은

$$
\mathbb{E}[X]
=
\sum_x x P(X=x)
$$

입니다.

공정한 주사위의 기대값을 계산해 봅니다.

In [ ]:
x = np.array([1, 2, 3, 4, 5, 6], dtype=float)
p = np.ones(6) / 6

expected_value = np.sum(x * p)

print("E[X] =", expected_value)

### 직접 해보기 4

주사위가 공정하지 않다고 가정하고 `p`를 바꾸어 보세요.

단, 확률은 다음을 만족해야 합니다.

```python
p.sum() == 1
```

확률이 큰 눈 쪽으로 기대값이 이동하는지 확인하세요.

## 16. Variance와 Covariance

Variance는 random variable이 평균 주변에서 얼마나 퍼져 있는지 나타냅니다.

$$
\mathrm{Var}(X)
=
\mathbb{E}
\left[
(X-\mathbb{E}[X])^2
\right]
$$

Covariance는 두 변수의 동시 변화 방향을 나타냅니다.

$$
\mathrm{Cov}(X,Y)
=
\mathbb{E}
\left[
(X-\mathbb{E}[X])
(Y-\mathbb{E}[Y])
\right]
$$

In [ ]:
rng = np.random.default_rng(0)

x = np.linspace(0, 10, 100)

# x가 증가할수록 함께 증가하는 y
y_positive = 2 * x + rng.normal(0, 2, size=len(x))

# x가 증가할수록 감소하는 y
y_negative = -2 * x + rng.normal(0, 2, size=len(x))

print("Variance of x:", np.var(x))
print("\nCovariance matrix: x vs y_positive")
print(np.cov(x, y_positive))

print("\nCovariance matrix: x vs y_negative")
print(np.cov(x, y_negative))

In [ ]:
plt.scatter(x, y_positive, label="Positive covariance")
plt.scatter(x, y_negative, label="Negative covariance")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Covariance")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### 확인할 내용

- 두 변수가 함께 증가하면 covariance는 양수입니다.
- 한 변수가 증가할 때 다른 변수가 감소하면 covariance는 음수입니다.
- covariance가 0에 가깝다는 것은 **선형적인 동시 변화가 약함**을 의미합니다.

# 17. 최종 실습

### 기본
1. `(3, 4)` matrix를 만들고 transpose한 뒤 shape를 확인하세요.
2. 두 벡터의 dot product와 outer product를 계산하고 결과 shape를 비교하세요.
3. 같은 두 matrix에 대해 `A @ B`와 `A * B`를 비교하세요.
4. broadcasting을 이용해 `(4, 3)` matrix의 모든 행에 동일한 bias vector를 더하세요.

### 비교
5. 하나의 vector에 대해 1-norm, 2-norm, infinity-norm을 비교하세요.
6. 세 쌍의 vector를 만들어 cosine similarity가 각각 1, 0, -1에 가까워지도록 해보세요.
7. `np.linalg.eig()`를 이용해 eigenpair를 구하고 `W @ x = lambda * x`를 검증하세요.

### 확률
8. 두 사건을 직접 정의하고 conditional probability를 계산하세요.
9. Bayes' theorem 예제에서 prior를 `1% → 10%`로 바꾸어 posterior가 어떻게 변하는지 확인하세요.
10. 확률분포를 직접 만들고 expectation과 variance를 계산하세요.

### 도전
11. 다음 함수들을 직접 구현해 보세요.

```python
def my_l1_norm(x):
    pass

def my_l2_norm(x):
    pass

def my_cosine_similarity(x, y):
    pass

def my_softmax(x):
    pass
```

NumPy의 내장 함수 또는 앞에서 구현한 함수와 결과가 같은지 비교하세요.

# 18. 정리

이번 실습에서는 AI에서 반복적으로 등장하는 수학 개념을 NumPy 연산과 연결했습니다.

| 수학 개념 | NumPy / AI에서의 연결 |
|---|---|
| Vector / Matrix / Tensor | 데이터와 파라미터 표현 |
| Transpose | 축 교환, 행렬 연산 |
| Dot Product | 유사도, 가중합 |
| Matrix Multiplication | Neural network layer |
| Element-wise Product | activation / gating 등의 원소별 연산 |
| Broadcasting | bias 연산 |
| Concatenation | feature 결합 |
| Linear / Affine Transformation | Dense layer |
| Eigenvalue / Eigenvector | 선형변환의 고유 방향 |
| Norm | 크기, 거리, regularization |
| Cosine Similarity | representation similarity |
| Determinant | 변환에 따른 면적/부피 변화 |
| Softmax | logits → probabilities |
| Conditional / Joint / Marginal Probability | 확률 모델링 |
| Bayes' Theorem | prior → posterior 업데이트 |
| Expectation | 확률적 평균 |
| Variance / Covariance | 분산과 변수 간 관계 |

### 꼭 기억할 것

AI에서 수학은 단순한 공식 암기보다 **배열의 shape와 연산이 무엇을 의미하는지 이해하는 것**이 중요합니다.

특히 앞으로 반복적으로 등장할 핵심 구조는 다음과 같습니다.

$$
\mathbf{y}
=
\mathbf{W}\mathbf{x}
+
\mathbf{b}
$$

즉, **입력 → 선형/affine 변환 → 다음 표현**이라는 구조입니다.